# Beispiel-Notebook: Nutzung des Time-Expanded Modells für Fracht-Routing

Dieses Notebook demonstriert, wie das high-level `TimeExpandedFreightRoutingModel` aus dem Paket `freight_routing` genutzt wird, um optimale Transportrouten in einem multimodalen Netzwerk zu berechnen.

---

In [ ]:
import sys
from pathlib import Path

# Projektpfad ermitteln und zu sys.path hinzufügen, damit 'freight_routing' importiert werden kann
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "freight_routing":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from freight_routing.data_loader import NetworkDataLoader  # noqa: E402
from freight_routing.data_models import Shipment, ObjectiveWeights  # noqa: E402
from freight_routing.model import TimeExpandedFreightRoutingModel  # noqa: E402
from freight_routing.visualization import create_network_map  # noqa: E402

print("Module erfolgreich importiert!")

Module erfolgreich importiert!


## 1. Laden des Transportnetzwerks

Wir laden ein kleines Test-Netzwerk (`test_network.json`), das drei Hubs (Berlin, Hamburg, München) und zwei Transportmodi (Straße und Schiene) enthält.

In [ ]:
test_network_path = PROJECT_ROOT / "dataset" / "test_network.json"
network_data = NetworkDataLoader.from_json(test_network_path)

# Zusammenfassung des geladenen Netzwerks anzeigen
network_data.summary()

Summary NetworkData:
hubs=3
arcs=6
modes=2


## 2. Definition der Sendungen (Shipments)

Eine Sendung (`Shipment`) beschreibt den Transportauftrag von einem Start-Hub zu einem Ziel-Hub.
Jede Sendung besitzt:
- Eine eindeutige ID (`id`)
- Start- und Zielort (`start_hub`, `end_hub`)
- Startzeitpunkt und späteste Lieferzeit (Deadline) in Minuten (`start_time`, `deadline`)
- Maximal zulässiges Budget (`max_price`)
- Optional ein Emissionslimit (`max_emissions`)
- Frachtgewicht in Tonnen (`weight`)

Wir definieren eine Sendung von **Berlin (BER)** nach **München (MUC)** mit einem Gewicht von 5 Tonnen, die ab Minute 0 bereitsteht und spätestens nach 48 Stunden (2880 Minuten) ankommen muss.

In [ ]:
shipment = Shipment(
    id="sendung_ber_muc_1",
    start_hub="BER",
    end_hub="MUC",
    start_time=0,
    deadline=2880,  # 48 Stunden Frist
    max_price=5000.0,  # Budgetgrenze
    max_emissions=None,  # Keine Emissionsgrenze
    weight=5.0,  # 5 Tonnen Gewicht
)

shipments = [shipment]
print(f"Definierte Sendung: {shipment}")

Definierte Sendung: Shipment(id='sendung_ber_muc_1', start_hub='BER', end_hub='MUC', start_time=0, deadline=2880, max_price=5000.0, max_emissions=None, weight=5.0)


## 3. Modell instanziieren, bauen und lösen

Wir instanziieren das Modell `TimeExpandedFreightRoutingModel` mit dem geladenen Netzwerk. 
Anschließend bauen wir das zeitexpandierte Netzwerk über einen Zeithorizont von z.B. 2 Tagen (`planning_days = 2`) und lösen das Optimierungsproblem für unsere Sendung.

In [ ]:
# 1. Modell instanziieren (standardmäßig mit gleichgewichteten Zielen)
model = TimeExpandedFreightRoutingModel(network_data)

# 2. Zeitexpandierten Graphen bauen (für 2 Planungstage)
model.build(planning_days=2, shipments=shipments)
model.summary()

# 3. Optimierungsproblem lösen
result = model.solve(shipments)

print("\n--- Optimierungsergebnisse ---")
print(f"Solver Status: {result.status}")
print(f"Optimalität erreicht: {result.is_optimal}")

Summary TimeExpandedFreightRoutingModel:
planning_days=2
planning_horizon_min=2880
nodes=83
arcs=119
  - transport_arcs=18
  - transfer_arcs=24
  - waiting_arcs=77

--- Optimierungsergebnisse ---
Solver Status: Optimal
Optimalität erreicht: True


## 4. Analyse der Ergebnisse und Routenverlauf

Wenn eine optimale Lösung gefunden wurde, können wir die Gesamtkosten, die Gesamtemissionen, die Gesamtdauer sowie den konkreten Routenverlauf im Detail auslesen.

In [ ]:
if result.is_optimal:
    print(
        f"Gesamtkosten: {result.total_cost:.2f} EUR (Fix: {result.total_fixed_cost:.2f} EUR, Variabel: {result.total_variable_cost:.2f} EUR)"
    )
    print(
        f"Gesamtemissionen: {result.total_emissions:.2f} kg CO2 (Fix: {result.total_fixed_emissions:.2f} kg, Variabel: {result.total_variable_emissions:.2f} kg)"
    )
    print(
        f"Gesamtzeit: {result.total_time} Minuten ({result.total_time / 60:.2f} Stunden)"
    )

    print("\nGewählter Routenverlauf:")
    route = result.shipment_routes[shipment.id]
    for i, arc in enumerate(route, 1):
        print(f"  Schritt {i}:")
        print(
            f"    Von: {arc.from_node.hub_id} ({arc.from_node.mode}) zur Zeit {arc.from_node.time_min} min"
        )
        print(
            f"    Nach: {arc.to_node.hub_id} ({arc.to_node.mode}) zur Zeit {arc.to_node.time_min} min"
        )
        print(
            f"    Typ: {arc.arc_type.value} | Distanz: {arc.distance} km | Modus: {arc.mode}"
        )
        print(
            f"    Kosten: {arc.cost:.2f} EUR | Emissionen: {arc.emissions:.2f} kg CO2"
        )
else:
    print(
        "Keine zulässige Route gefunden, die alle Nebenbedingungen (z.B. Deadlines) einhält."
    )

Gesamtkosten: 960.00 EUR (Fix: 200.00 EUR, Variabel: 760.00 EUR)
Gesamtemissionen: 382.00 kg CO2 (Fix: 20.00 kg, Variabel: 362.00 kg)
Gesamtzeit: 1260.0 Minuten (21.00 Stunden)

Gewählter Routenverlauf:
  Schritt 1:
    Von: BER (road) zur Zeit 0 min
    Nach: BER (road) zur Zeit 360 min
    Typ: waiting | Distanz: 0.0 km | Modus: road
    Kosten: 12.00 EUR | Emissionen: 0.30 kg CO2
  Schritt 2:
    Von: BER (road) zur Zeit 360 min
    Nach: HAM (road) zur Zeit 600 min
    Typ: transport | Distanz: 290.0 km | Modus: road
    Kosten: 43.50 EUR | Emissionen: 23.20 kg CO2
  Schritt 3:
    Von: HAM (road) zur Zeit 600 min
    Nach: HAM (road) zur Zeit 720 min
    Typ: waiting | Distanz: 0.0 km | Modus: road
    Kosten: 5.00 EUR | Emissionen: 0.10 kg CO2
  Schritt 4:
    Von: HAM (road) zur Zeit 720 min
    Nach: MUC (road) zur Zeit 1260 min
    Typ: transport | Distanz: 610.0 km | Modus: road
    Kosten: 91.50 EUR | Emissionen: 48.80 kg CO2


## 5. Kosten vs. Emissionen: Gewichtung anpassen

Über das Objekt `ObjectiveWeights` können wir festlegen, wie stark die einzelnen Zielkriterien (Kosten, Emissionen, Zeit) gewichtet werden.

Wir vergleichen zwei Szenarien für unsere Sendung:
1. **Szenario A: Reine Kostenoptimierung** (Kostengewicht = 1.0, Emissionsgewicht = 0.0)
2. **Szenario B: Reine Emissionsoptimierung** (Kostengewicht = 0.0, Emissionsgewicht = 1.0)

In [ ]:
# Szenario A: Kosten im Fokus
weights_cost = ObjectiveWeights(cost=1.0, emissions=0.0, time=0.0)
model_cost = TimeExpandedFreightRoutingModel(
    network_data, objective_weights=weights_cost
)
model_cost.build(planning_days=2, shipments=shipments)
result_cost = model_cost.solve(shipments)

# Szenario B: Emissionen im Fokus
weights_eco = ObjectiveWeights(cost=0.0, emissions=1.0, time=0.0)
model_eco = TimeExpandedFreightRoutingModel(network_data, objective_weights=weights_eco)
model_eco.build(planning_days=2, shipments=shipments)
result_eco = model_eco.solve(shipments)

print("=== Szenario A: Kostenoptimiert ===")
if result_cost.is_optimal:
    print(f"Kosten: {result_cost.total_cost:.2f} EUR")
    print(f"Emissionen: {result_cost.total_emissions:.2f} kg CO2")
    route_cost = result_cost.shipment_routes[shipment.id]
    print(
        f'Route: {" -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in route_cost)} -> {route_cost[-1].to_node.hub_id}'
    )
else:
    print("Keine Lösung")

print("\n=== Szenario B: Emissionsoptimiert ===")
if result_eco.is_optimal:
    print(f"Kosten: {result_eco.total_cost:.2f} EUR")
    print(f"Emissionen: {result_eco.total_emissions:.2f} kg CO2")
    route_eco = result_eco.shipment_routes[shipment.id]
    print(
        f'Route: {" -> ".join(f"{arc.from_node.hub_id}({arc.mode})" for arc in route_eco)} -> {route_eco[-1].to_node.hub_id}'
    )
else:
    print("Keine Lösung")

=== Szenario A: Kostenoptimiert ===
Kosten: 960.00 EUR
Emissionen: 382.00 kg CO2
Route: BER(road) -> BER(road) -> HAM(road) -> HAM(road) -> MUC

=== Szenario B: Emissionsoptimiert ===
Kosten: 1170.00 EUR
Emissionen: 191.88 kg CO2
Route: BER(rail) -> BER(rail) -> HAM(rail) -> HAM(rail) -> HAM(rail) -> HAM(rail) -> HAM(rail) -> MUC


## 6. Routen-Visualisierung auf einer interaktiven Karte

Mit der Funktion `create_network_map` können wir die static Verbindungen des multimodalen Netzwerks sowie die berechnete Route auf einer interaktiven Folium-Karte darstellen.

In [ ]:
if result_eco.is_optimal:
    # Generiere Karte für die emissionsoptimierte Route
    m = create_network_map(
        network_data, route=result_eco.shipment_routes[shipment.id], show_network=True
    )
    # Karte im Notebook anzeigen (oder mit m.save('route.html') als Datei speichern)
    display(m)
else:
    print("Keine Route zur Visualisierung verfügbar.")